In [1]:
# Librerias necesarias
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from copy import deepcopy
from generacion_pacientes import generar_pacientes
from collections import deque
# # Add the parent directory to sys.path to allow relative imports
# sys.path.append(os.path.abspath(os.path.join('..', '1. codigo analisis')))
import parametros as p
from kpis import * # Mala practica pero facil
from clases import Paciente, Simulacion, ModeloA, ModeloBase, ModeloProactivo

In [2]:
def correr_multiples_simulaciones(
    clase_modelo_base,
    T_max=4500,
    ciclos=4208,
    num_simulaciones=5,
    seed_inicial=0,
    clase_modelo_alternativo=None,
    ciclo_de_cambio=0,
    pacientes_caso_base=False,
    log_detallado=True
):
    # Nombres base y alternativo para la carpeta principal
    nombre_base = clase_modelo_base.__name__
    nombre_alternativo = clase_modelo_alternativo.__name__ if clase_modelo_alternativo else "None"
    nombre_carpeta = f"{nombre_base}_{nombre_alternativo}_T{T_max}_C{ciclos}"
    
    # Crear carpeta principal y subcarpetas
    base_dir = os.path.join("resultados simulacion", nombre_carpeta)
    logs_dir = os.path.join(base_dir, "logs")
    plots_dir = os.path.join(base_dir, "plots")
    kpis_dir = os.path.join(base_dir, "kpis")
    os.makedirs(logs_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)
    os.makedirs(kpis_dir, exist_ok=True)

    for i in range(num_simulaciones):
        seed = seed_inicial + i
        print(f"\n⏳ Simulación {i+1}/{num_simulaciones} con seed={seed}")

        # Reiniciar contador global
        Paciente.CONTADOR_ID = 1

        # Instanciar modelos
        modelo = clase_modelo_base()
        alternativo = clase_modelo_alternativo() if clase_modelo_alternativo else None

        # Ejecutar simulación
        simu = Simulacion(
            T_max, seed, ciclos,
            modelo=modelo,
            modelo_alternativo=alternativo,
            pacientes_caso_base=pacientes_caso_base,
            ciclo_de_cambio=ciclo_de_cambio,
            log_detallado=log_detallado
        )
        df = simu.simular()

        # Calcular KPIs
        t0 = time.time()
        kpis = calcular_kpis(
            df,
            save_plot=True,
            modelo=modelo,
            seed=seed,
            ciclos=ciclos,
            save_dir=plots_dir
        )
        print(f"✅ KPIs calculados en {time.time() - t0:.2f} segundos")

        # Guardar log como CSV
        filename_prefix = f"{seed}"
        ruta_csv = os.path.join(logs_dir, f"{filename_prefix}.csv")
        df.to_csv(ruta_csv, index=False)
        print(f"📄 Log guardado en: {ruta_csv}")

        # Guardar KPIs como JSON
        ruta_json = os.path.join(kpis_dir, f"{filename_prefix}.json")
        with open(ruta_json, 'w') as f:
            json.dump(kpis, f, indent=4)
        print(f"📊 KPIs guardados en: {ruta_json}")

In [3]:
correr_multiples_simulaciones(
    clase_modelo_base=ModeloProactivo,
    T_max=4500,
    ciclos=4208,
    num_simulaciones=1,
    seed_inicial=1,
    clase_modelo_alternativo=None,
    pacientes_caso_base=False,
    ciclo_de_cambio=0
)


⏳ Simulación 1/1 con seed=1
Se utiliza archivo existente 1_4208.json de pacientes (0.00 segundos)
Pacientes separados por llegada cargados (3.88 segundos)
Clase Simulacion instanciada (0.00 segundos)
GA quiebra stock 173.74927887800038, h1: 15, h2: 15, h3: 19
GA quiebra stock 432.9601899999984, h1: 15, h2: 14, h3: 17
GA quiebra stock 1015.3237689999969, h1: 19, h2: 11, h3: 19
GA quiebra stock 425.8055143379993, h1: 14, h2: 16, h3: 8
GA quiebra stock 283.20893960800083, h1: 17, h2: 19, h3: 5
GA quiebra stock 139.05653089999623, h1: 12, h2: 12, h3: 16
GA quiebra stock 958.5450824999994, h1: 15, h2: 7, h3: 16
GA quiebra stock 219.10358517000077, h1: 13, h2: 6, h3: 16
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con grd 8
paciente 66403 en unidad 3 con